# Retrieve SIH hospitalization records

This notebook extracts hospitalization records from the Brazilian Hospital Information System (SIH) available through Base dos Dados.

The resulting dataset is used to analyze hospitalizations related to gambling and betting disorders and to compare hospitalization trends with outpatient visits over time in a line chart.

In [15]:
import os
import basedosdados as bd

In [16]:
import os
import basedosdados as bd

billing_id = os.getenv("billing_id")

query_sih = """
WITH internacoes AS (
    SELECT
        dados.ano,
        dados.mes,
        dados.sigla_uf,

        diretorio_uf.nome AS nome_uf,

        dados.id_municipio_estabelecimento_aih,
        diretorio_municipio_estabelecimento.nome
            AS nome_municipio_estabelecimento,

        dados.id_aih,

        dados.ano_internacao,
        dados.mes_internacao,
        dados.data_entrada_internacao,
        dados.data_saida_iternacao AS data_saida_internacao,

        dados.id_municipio_paciente,
        diretorio_municipio_paciente.nome
            AS nome_municipio_paciente,

        dados.id_estabelecimento_cnes,
        dados.id_procedimento_principal,

        dados.id_cid_principal,
        dados.id_cid_secundario,

        dados.indicador_uf_paciente

    FROM `basedosdados.br_ms_sih.servicos_profissionais` AS dados

    LEFT JOIN (
        SELECT DISTINCT
            sigla,
            nome
        FROM `basedosdados.br_bd_diretorios_brasil.uf`
    ) AS diretorio_uf
        ON dados.sigla_uf = diretorio_uf.sigla

    LEFT JOIN (
        SELECT DISTINCT
            id_municipio,
            nome
        FROM `basedosdados.br_bd_diretorios_brasil.municipio`
    ) AS diretorio_municipio_estabelecimento
        ON dados.id_municipio_estabelecimento_aih
            = diretorio_municipio_estabelecimento.id_municipio

    LEFT JOIN (
        SELECT DISTINCT
            id_municipio,
            nome
        FROM `basedosdados.br_bd_diretorios_brasil.municipio`
    ) AS diretorio_municipio_paciente
        ON dados.id_municipio_paciente
            = diretorio_municipio_paciente.id_municipio

    WHERE dados.ano BETWEEN 2015 AND 2025
      AND (
          dados.id_cid_principal IN ('F63', 'Z72')
          OR dados.id_cid_secundario IN ('F63', 'Z72')
      )
),

cnes AS (
    SELECT
        ano,
        mes,
        id_estabelecimento_cnes,
        id_municipio AS id_municipio_cnes,
        cep,
        indicador_vinculo_sus

    FROM `basedosdados.br_ms_cnes.estabelecimento`

    WHERE ano BETWEEN 2015 AND 2025
)

SELECT
    internacoes.*,

    cnes.id_municipio_cnes,
    diretorio_municipio_cnes.nome AS nome_municipio_cnes,

    cnes.cep,
    cnes.indicador_vinculo_sus

FROM internacoes

LEFT JOIN cnes
    ON internacoes.ano = cnes.ano
    AND internacoes.mes = cnes.mes
    AND internacoes.id_estabelecimento_cnes
        = cnes.id_estabelecimento_cnes

LEFT JOIN (
    SELECT DISTINCT
        id_municipio,
        nome
    FROM `basedosdados.br_bd_diretorios_brasil.municipio`
) AS diretorio_municipio_cnes
    ON cnes.id_municipio_cnes
        = diretorio_municipio_cnes.id_municipio
"""

df_sih = bd.read_sql(
    query=query_sih,
    billing_project_id=billing_id
)

df_sih

Downloading: 100%|██████████|


,ano,mes,sigla_uf,nome_uf,id_municipio_estabelecimento_aih,nome_municipio_estabelecimento,id_aih,ano_internacao,mes_internacao,data_entrada_internacao,...,nome_municipio_paciente,id_estabelecimento_cnes,id_procedimento_principal,id_cid_principal,id_cid_secundario,indicador_uf_paciente,id_municipio_cnes,nome_municipio_cnes,cep,indicador_vinculo_sus
0,2024,10,SP,São Paulo,3551009,São Vicente,3524126285559,2024,10,2024-10-02,...,None,7371349,0303170131,F63,None,0,3551009,São Vicente,11349000,1
1,2024,10,SP,São Paulo,3551009,São Vicente,3524126285559,2024,10,2024-10-02,...,None,7371349,0303170131,F63,None,0,3551009,São Vicente,11349000,1
2,2024,10,SP,São Paulo,3551009,São Vicente,3524126285559,2024,10,2024-10-02,...,None,7371349,0303170131,F63,None,0,3551009,São Vicente,11349000,1
3,2024,10,SP,São Paulo,3551009,São Vicente,3524126285559,2024,10,2024-10-02,...,None,7371349,0303170131,F63,None,0,3551009,São Vicente,11349000,1
4,2024,10,SP,São Paulo,3551009,São Vicente,3524126285416,2024,10,2024-10-02,...,None,7371349,0303170131,F63,None,0,3551009,São Vicente,11349000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
612,2025,10,SP,São Paulo,3515707,Ferraz de Vasconcelos,3525129197810,2025,10,2025-08-25,...,None,2080079,0303170140,F63,None,1,3515707,Ferraz de Vasconcelos,08502200,1
613,2025,10,SP,São Paulo,3515707,Ferraz de Vasconcelos,3525129197810,2025,10,2025-08-25,...,None,2080079,0303170140,F63,None,1,3515707,Ferraz de Vasconcelos,08502200,1
614,2025,10,SP,São Paulo,3515707,Ferraz de Vasconcelos,3525129197810,2025,10,2025-08-25,...,None,2080079,0303170140,F63,None,1,3515707,Ferraz de Vasconcelos,08502200,1
615,2025,10,SP,São Paulo,3515707,Ferraz de Vasconcelos,3525129581336,2025,10,2025-10-11,...,None,2080079,0303170131,F63,None,0,3515707,Ferraz de Vasconcelos,08502200,1


In [17]:
df_sih.shape

(617, 22)

In [18]:
df_sih["ano"].unique()

<IntegerArray>
[2024, 2025, 2021, 2023, 2020, 2022, 2019]
Length: 7, dtype: Int64

In [19]:
df_sih.groupby("sigla_uf").size()

sigla_uf
AL     10
CE      8
GO     12
MG      6
PA     27
PB     18
PE     28
PR      4
RJ     14
RO      7
RS     46
SC      4
SE      3
SP    412
TO     18
dtype: int64

In [20]:
df_sih.groupby("ano").size()

ano
2019     59
2020     22
2021      8
2022     67
2023     52
2024    266
2025    143
dtype: int64